In [29]:
import os
import sys
import yaml
import pandas as pd
import numpy as np

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import writing_tools as wt
from utils import stars_p

from matplotlib import pyplot as plt
from scipy.stats import f_oneway

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

os.environ['LOKY_MAX_CPU_COUNT'] = '1' # because of windows core count warning

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10

In [30]:
df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data_w_embeddings.parquet"))
reasons_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "reasons.parquet"))


In [31]:
df['outcome'] = df['project_result'].map({
    'APPROVED': 'Approved',
    'APPROVED WITH MINOR CONDITIONS OR MODIFICATIONS': 'Approved w/ conditions',
    'APPROVED WITH MAJOR CONDITIONS OR MODIFICATIONS': 'Approved w/ conditions',
    'DENIED': 'Delayed / Denied',
    'APPLICATION WITHDRAWN': 'Delayed / Denied',
    'DELAYED': 'Delayed / Denied'
})
n_approved = f"{(df['outcome'] == "Approved").sum():,.0f}"
n_approved_w_conditions = f"{(df['outcome'] == "Approved w/ conditions").sum():,.0f}"
n_delayed_denied = f"{(df['outcome'] == "Delayed / Denied").sum():,.0f}"

In [32]:
def get_descriptive_stats_continuous(variable):
    # Get descriptive stats for continuous variables
    approved_mask = df['outcome'] == "Approved"
    approved_w_conditions_mask = df['outcome'] == "Approved w/ conditions"
    delayed_denied_mask = df['outcome'] == "Delayed / Denied"
    
    # Calculate mean and standard deviation for approved cases
    approved_mean = df.loc[approved_mask, variable].mean()
    approved_std = df.loc[approved_mask, variable].std()
    
    # Calculate mean and standard deviation for approved w/ conditions cases
    approved_w_conditions_mean = df.loc[approved_w_conditions_mask, variable].mean()
    approved_w_conditions_std = df.loc[approved_w_conditions_mask, variable].std()

    # Calculate mean and standard deviation for delayed/denied cases
    delayed_denied_mean = df.loc[delayed_denied_mask, variable].mean()
    delayed_denied_std = df.loc[delayed_denied_mask, variable].std()
    
    # Calculate F-test for mean differences between groups
    f_stat, p_value = f_oneway(
        df.loc[approved_mask, variable],
        df.loc[approved_w_conditions_mask, variable],
        df.loc[delayed_denied_mask, variable]
    )
    
    return approved_mean, approved_w_conditions_mean, delayed_denied_mean, f_stat, p_value

In [33]:
header = fr"""
\begin{{table}}[H]
\centering
\caption{{Descriptive Statistics by Motion Outcome}}
\vspace{{0.2cm}}
\label{{tab_bivariate_descriptives}}
\begin{{adjustbox}}{{max width=\textwidth}}
\begin{{threeparttable}}
\begin{{tabular}}{{lrrrll}} \toprule
 Variable & \makecell{{Approved \\ ($n = {n_approved}$)}} & \makecell{{Approved w/ \\ conditions \\ ($n = {n_approved_w_conditions}$)}} & \makecell{{Delayed / \\ denied \\ ($n = {n_delayed_denied}$)}} & \makecell{{Test stat}} & \makecell{{p-value}}  \\ \midrule
 & & & & & \\
"""
footer = r"""
\bottomrule
\end{tabular}
\begin{tablenotes}
\item {\textit{Notes: } This reports descriptive statistics for variables grouped by motion outcome. To test differences between groups, a one-way ANOVA F-test was conducted for continuous variables and a chi-square test was conducted for categorical variables.}
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}
"""


In [34]:
public_input = r"""
\textit{Public input} & & & & & \\
"""

var = "n_docs"
varname = "~ ~ Number of letters (mean)"
results = get_descriptive_stats_continuous(var)
public_input += rf"{varname} & {results[0]:.1f} & {results[1]:.1f} & {results[2]:.1f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"

var = "n_support"
varname = "~ ~ ~ ~ in support"
results = get_descriptive_stats_continuous(var)
public_input += rf"{varname} & {results[0]:.1f} & {results[1]:.1f} & {results[2]:.1f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"

var = "n_oppose"
varname = "~ ~ ~ ~ in opposition"
results = get_descriptive_stats_continuous(var)
public_input += rf"{varname} & {results[0]:.1f} & {results[1]:.1f} & {results[2]:.1f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"


print(public_input)





\textit{Public input} & & & & & \\
~ ~ Number of letters (mean) & 4.4 & 10.0 & 11.1 & $F = 6.45$ & $0.002^{***}$ \\
~ ~ ~ ~ in support & 1.9 & 4.7 & 3.3 & $F = 3.64$ & $0.027^{**}$ \\
~ ~ ~ ~ in opposition & 2.1 & 4.5 & 7.0 & $F = 7.47$ & $0.001^{***}$ \\



In [35]:
with open(os.path.join(LOCAL_PATH, "tables", "tab_bivariate_descriptives.tex"), "w", encoding='utf-8') as f:
    f.write(header + public_input + footer)


In [36]:
reasons_df

,date,year,doc_id,item_no,support_or_oppose,reason
0,2018-05-10,2018,2,6,DEFINITELY SUPPORT,housing supply / affordability
1,2018-05-10,2018,2,6,DEFINITELY SUPPORT,homelessness
2,2018-05-10,2018,2,6,DEFINITELY SUPPORT,transit access / walkability
3,2018-05-10,2018,2,6,DEFINITELY SUPPORT,CEQA related matters
4,2018-05-10,2018,2,6,DEFINITELY SUPPORT,procedural issues
...,...,...,...,...,...,...
24217,2026-02-26,2026,53,6,SOMEWHAT OPPOSE,climate / environment
24218,2026-02-26,2026,53,6,SOMEWHAT OPPOSE,safety impact
24219,2026-02-26,2026,53,6,SOMEWHAT OPPOSE,displacement / gentrification
24220,2026-02-26,2026,54,7,DEFINITELY SUPPORT,CEQA related matters
